# 250 · A continuous wave inside an experiment — solution

**Level:** medium · **Topics:** Gen2, SHFQC, dsl, pulses, sweeps, spectroscopy, calibration · **Type:** write the code

Solution of [exercise.ipynb](exercise.ipynb).

In [ ]:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from laboneq.simple import *

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## The experiment

Three things are needed:

1. a constant pulse of `length_of_step` that fits in the memory of the instrument, whatever its length,
2. a sweep of the **frequency of the drive oscillator**: the oscillator is part of the calibration, so the sweep parameter goes into the calibration of the experiment,
3. the measurement at the end of each step.

📖 [Pulse commands](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/00_pulse_commands.html) ·
[Frequency sweep examples](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/03_frequency_sweep_examples.html) ·
[Experiment calibration](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/00_experiment_calibration.html) ·
[Timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html)

In [ ]:
def drive_cw(frequency_sweep, length_of_step, integration_length):
    cw_pulse = pulse_library.const(length=length_of_step, amplitude=0.5, can_compress=True)
    readout_pulse = pulse_library.const(length=integration_length, amplitude=0.5)

    @dsl.experiment(signals=["drive", "measure", "acquire"])
    def cw_sweep():
        with dsl.acquire_loop_rt(count=1):
            with dsl.sweep(parameter=frequency_sweep) as frequency:
                with dsl.section(name="step", alignment=SectionAlignment.RIGHT):
                    dsl.play("drive", cw_pulse)
                    dsl.measure(
                        measure_signal="measure",
                        acquire_signal="acquire",
                        measure_pulse=readout_pulse,
                        integration_kernel=readout_pulse,
                        handle="result",
                    )

        experiment_calibration = dsl.experiment_calibration()
        experiment_calibration["drive"] = SignalCalibration(
            oscillator=Oscillator(
                frequency=frequency, modulation_type=ModulationType.HARDWARE
            )
        )

    experiment = cw_sweep()
    experiment.set_signal_map(
        {
            "drive": "q0/drive",
            "measure": "q0/measure",
            "acquire": "q0/acquire",
        }
    )
    return experiment

## Run it

101 frequencies, 1 ms per step, 2 µs of integration.

In [ ]:
frequencies = LinearSweepParameter(start=-200e6, stop=200e6, count=101)

experiment = drive_cw(frequencies, length_of_step=1e-3, integration_length=2e-6)
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)
print(round(compiled.estimated_runtime, 2), "s")

**Expected result:**

```
(101,)
0.1 s
```

## Look at it

With short steps of 10 µs the simulation shows the first two steps: the drive is on during the whole step,
and the readout sits at the end of each step.

In [ ]:
short = session.compile(drive_cw(frequencies, length_of_step=10e-6, integration_length=1e-6))
plot_simulation(short, start_time=0, length=22e-6, signals=["drive", "measure"])

## Answers

1. No. The drive is off for about 90 ns between two steps, while the oscillator changes frequency, and the phase is not continuous from one step to the next. For a slow spectroscopy this does not matter. For anything that relies on a steady state, it does.
2. 2.048 µs (4096 samples). Longer integrations need the LRT option: the compiler says so (`Acquisition length ... exceeds ... does not have the LRT option installed`).
3. It compiles and the instrument memory is not an issue, thanks to `can_compress=True`: without it, even the 1 ms step fails with `waveforms are not fitting into wave memory`. But the compilation gets slow: for 101 steps of 0.6 s (one minute in total) it took more than a minute on a laptop, and steps of 10 s did not finish.
4. It is possible, with three limits to know: short gaps at every frequency change, an integration of at most 2 µs without LRT, and a compilation time that grows with the length of the experiment. If a true CW is needed, the instrument has to be set outside of the real-time experiment, for example in a near-time sweep.

**Takeaway:** a compressed constant pulse and a swept hardware oscillator make a good pseudo-CW. It is a sequence of long pulses, not a continuous wave.